# Revision experiments (IJEEEMI reviewer comments)

Runs the five experiments that need a GPU. It reuses the trained models saved in your Drive (`IJEEEMI_CXR/ckpt`).

| Part | Reviewer comment | Approx. time on a free T4 |
|---|---|---|
| A | R2-a: class-weighted and balanced-sampling training (3 models x 3 seeds x 2) | ~50 min |
| B | R1-8: Monte Carlo dropout (MobileNetV3, EfficientNet-B0) | ~25 min |
| C | R2-d: test-time augmentation count ablation (K = 2, 4, 8, 16) | ~20 min |
| D | R1-9: size, FLOPs, latency, throughput, memory, energy; GPU and 2-thread CPU | ~10 min |
| E | R2-f: EfficientNet-B0 numerical diagnostics (fp16 / fp32 / fp64, activations) | ~5 min |

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then **Runtime -> Run all**. Every part saves its results to Drive and is skipped if already finished, so after a disconnect simply **Run all** again.
At the end, download **`IJEEEMI_CXR/revision_results.zip`** from Drive and upload it to the chat.

In [ ]:
# ===== R0. Setup =====
import os, sys, json, time, subprocess, threading
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)
ROOT = "/content/drive/MyDrive/IJEEEMI_CXR"
REV = os.path.join(ROOT, "revision")
for sub in ["ckpt", "preds", "tables"]:
    os.makedirs(os.path.join(REV, sub), exist_ok=True)
CFG = json.load(open(os.path.join(ROOT, "logs", "config.json")))
TEMPS = json.load(open(os.path.join(ROOT, "tables", "temperatures.json")))
missing = [k for k in TEMPS if not os.path.exists(os.path.join(ROOT, "ckpt", k + "_best.pt"))]
assert not missing, f"Original checkpoints missing in Drive: {missing}"
print("Found all 9 original checkpoints. Results will be saved in", REV)

In [ ]:
%%writefile core.py
# ===== Physics-informed degradations, metrics, and conformal risk control =====
# All degradations act on display images v in [0,1] (bright = high attenuation),
# which we map to a line-integral proxy p = v * P_MAX (Beer-Lambert: I = I0 * exp(-p)).
import numpy as np
from scipy.ndimage import gaussian_filter, convolve1d, zoom
from sklearn.metrics import roc_auc_score

P_MAX = 4.0        # max line integral represented by v = 1
N0 = 1.0e4         # photons per pixel (unattenuated) at nominal (full) dose
SIGMA_E = 5.0      # electronic (Gaussian) noise, in photon-count units

DEGRADATIONS = {
    # name: (list of 5 severity parameters, human-readable parameter name)
    "dose":     ([0.5, 0.25, 0.1, 0.05, 0.02], "dose fraction d"),
    "scatter":  ([0.25, 0.5, 1.0, 2.0, 4.0],   "scatter-to-primary ratio"),
    "contrast": ([0.85, 0.7, 0.55, 0.4, 0.25], "subject-contrast factor kappa"),
    "overexp":  ([1.5, 2.0, 3.0, 5.0, 8.0],    "detector gain error e"),
    "motion":   ([3, 5, 9, 13, 19],            "motion length (px)"),
}

def _to_p(v):   return v.astype(np.float32) * P_MAX
def _to_v(p):   return np.clip(p / P_MAX, 0.0, 1.0).astype(np.float32)

def deg_dose(v, d, rng):
    """Quantum (Poisson) + electronic (Gaussian) noise at dose fraction d."""
    p = _to_p(v)
    lam = d * N0 * np.exp(-p)
    n = rng.poisson(lam).astype(np.float32) + rng.normal(0, SIGMA_E, size=lam.shape).astype(np.float32)
    n = np.maximum(n, 1.0)
    return _to_v(-np.log(n / (d * N0)))

def deg_scatter(v, spr, rng=None):
    """Additive low-frequency scatter S with mean(S)/mean(I_p) = SPR (fixed display window)."""
    p = _to_p(v)
    ip = np.exp(-p)                                   # (B,H,W), I0 = 1
    B, H, W = ip.shape
    f = 8                                             # blur on a downsampled grid (fast, equivalent for a wide kernel)
    small = zoom(ip, (1, 1 / f, 1 / f), order=1)
    small = gaussian_filter(small, sigma=(0, H / (8 * f), W / (8 * f)), mode="reflect")
    s = zoom(small, (1, H / small.shape[1], W / small.shape[2]), order=1)
    s = s * (spr * ip.mean(axis=(1, 2), keepdims=True) / np.maximum(s.mean(axis=(1, 2), keepdims=True), 1e-8))
    return _to_v(-np.log(ip + s))

def deg_contrast(v, kappa, rng=None):
    """kVp-like subject-contrast reduction: p' = mean(p) + kappa (p - mean(p))."""
    p = _to_p(v)
    m = p.mean(axis=(1, 2), keepdims=True)
    return _to_v(m + kappa * (p - m))

def deg_overexp(v, e, rng=None):
    """Detector gain error e > 1 with saturation at I_sat = 1, reconstructed with nominal calibration."""
    ip = np.exp(-_to_p(v))
    idet = np.minimum(e * ip, 1.0)
    return _to_v(-np.log(np.maximum(idet, 1e-8)))

def deg_motion(v, L, rng=None):
    """Vertical (respiratory-like) linear motion blur of length L pixels."""
    k = np.ones(int(L), dtype=np.float32) / float(L)
    return convolve1d(v.astype(np.float32), k, axis=1, mode="reflect")

DEG_FN = {"dose": deg_dose, "scatter": deg_scatter, "contrast": deg_contrast,
          "overexp": deg_overexp, "motion": deg_motion}

def conditions():
    out = [("clean", 0, None)]
    for name, (params, _) in DEGRADATIONS.items():
        for i, prm in enumerate(params):
            out.append((name, i + 1, prm))
    return out

def apply_condition(v, name, prm, seed):
    """v: float32 (B,H,W) in [0,1]. Deterministic given seed."""
    if name == "clean":
        return v.astype(np.float32)
    rng = np.random.default_rng(seed)
    return DEG_FN[name](v, prm, rng)

def noise_estimate(v):
    """Immerkaer (1996) fast noise sigma estimate, per image. Model-free quality score."""
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    from scipy.ndimage import convolve
    out = np.empty(v.shape[0], dtype=np.float32)
    H, W = v.shape[1:]
    for i in range(v.shape[0]):
        r = convolve(v[i], k, mode="reflect")
        out[i] = np.sqrt(np.pi / 2) * np.abs(r).sum() / (6 * (W - 2) * (H - 2))
    return out

# ---------------- metrics (binary, single logit) ----------------
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def ece_binary(prob_pos, y, n_bins=15):
    pred = (prob_pos >= 0.5).astype(int)
    conf = np.where(pred == 1, prob_pos, 1 - prob_pos)
    correct = (pred == y).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins[1:-1], right=True), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return e

def metrics_from_logits(logits, y, T=1.0, conf_thr=0.9):
    p = sigmoid(logits / T)
    pred = (p >= 0.5).astype(int)
    conf = np.where(pred == 1, p, 1 - p)
    wrong = pred != y
    eps = 1e-7
    out = {
        "acc": float((~wrong).mean()),
        "bal_acc": float(0.5 * ((pred[y == 1] == 1).mean() + (pred[y == 0] == 0).mean())),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "ece": float(ece_binary(p, y)),
        "brier": float(np.mean((p - y) ** 2)),
        "nll": float(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))),
        "conf_err_rate": float((wrong & (conf >= conf_thr)).mean()),
        "frac_err_confident": float((wrong & (conf >= conf_thr)).sum() / max(wrong.sum(), 1)),
        "mean_conf_wrong": float(conf[wrong].mean()) if wrong.any() else np.nan,
        "sens": float((pred[y == 1] == 1).mean()),
        "spec": float((pred[y == 0] == 0).mean()),
    }
    return out

# ---------------- conformal risk control (Angelopoulos et al., 2024) ----------------
def crc_threshold(scores_cal, loss_fn_cal, alpha, B=1.0):
    """
    Units i = 1..n (base images). Gate ACCEPTS a sample when score <= lam.
    loss_fn_cal(lam) -> array (n,) of per-unit losses in [0,B], non-decreasing in lam.
    Returns largest lam with (n * Rhat(lam) + B) / (n + 1) <= alpha  (or -inf: reject all).
    """
    n = loss_fn_cal(np.inf).shape[0]
    grid = np.unique(scores_cal.ravel())
    best = -np.inf
    # risk is monotone in lam -> binary search over grid
    lo, hi = 0, len(grid) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r = (n * loss_fn_cal(grid[mid]).mean() + B) / (n + 1)
        if r <= alpha:
            best = grid[mid]; lo = mid + 1
        else:
            hi = mid - 1
    return best

def unit_loss(scores, conf_err, lam):
    """scores, conf_err: (n_units, n_conditions). Loss = fraction of a unit's versions that are accepted confident errors."""
    return ((scores <= lam) & conf_err).mean(axis=1)

def run_crc(scores, conf_err, alpha, n_trials=200, cal_frac=0.5, seed=0):
    """scores, conf_err: (n_units, n_cond). Random image-level splits. Returns list of dicts."""
    rng = np.random.default_rng(seed)
    n = scores.shape[0]
    res = []
    for t in range(n_trials):
        perm = rng.permutation(n)
        nc = int(cal_frac * n)
        c, te = perm[:nc], perm[nc:]
        lam = crc_threshold(scores[c], lambda l: unit_loss(scores[c], conf_err[c], l), alpha)
        acc_mask = scores[te] <= lam
        res.append({
            "trial": t, "lambda": float(lam),
            "test_risk": float((acc_mask & conf_err[te]).mean()),
            "accept_rate": float(acc_mask.mean()),
            "risk_no_gate": float(conf_err[te].mean()),
            "cond_conf_err_among_accepted": float((acc_mask & conf_err[te]).sum() / max(acc_mask.sum(), 1)),
        })
    return res


In [ ]:
# ===== R1. Data =====
import numpy as np, importlib, core, pandas as pd
importlib.reload(core)
from medmnist import PneumoniaMNIST
def load(split):
    ds = PneumoniaMNIST(split=split, download=True, size=224, root=CFG["DATA_DIR"])
    X = np.asarray(ds.imgs)
    if X.ndim == 4: X = X[..., 0]
    return X.astype(np.uint8), np.asarray(ds.labels).reshape(-1).astype(np.int64)
Xtr, ytr = load("train"); Xva, yva = load("val"); Xte, yte = load("test")
assert len(Xte) == 624, "Test set must be the full 624 images"
V_test = Xte.astype(np.float32) / 255.0
CONDS = core.conditions()
print(len(Xtr), len(Xva), len(Xte), "pneumonia fraction (train) =", ytr.mean().round(3))

In [ ]:
# ===== 3. Models, augmentation, prediction helpers =====
import math, torch.nn as nn, torch.nn.functional as F, torchvision.models as tvm
DEV = "cuda"
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

def build(name):
    if name == "mobilenet_v3_large":
        m = tvm.mobilenet_v3_large(weights="DEFAULT"); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights="DEFAULT"); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "resnet18":
        m = tvm.resnet18(weights="DEFAULT"); m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "densenet121":
        m = tvm.densenet121(weights="DEFAULT"); m.classifier = nn.Linear(m.classifier.in_features, 1)
    else:
        raise ValueError(name)
    return m.to(DEV)

def to_input(x):                       # x: (B,1,H,W) in [0,1]
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

def random_affine(x, rot_deg, trans, scale, gen=None):
    B = x.shape[0]
    r = lambda: torch.rand(B, device=x.device, generator=gen) * 2 - 1
    a = r() * rot_deg * math.pi / 180; s = 1 + r() * scale; tx = r() * trans * 2; ty = r() * trans * 2
    theta = torch.stack([torch.stack([torch.cos(a) / s, -torch.sin(a) / s, tx], 1),
                         torch.stack([torch.sin(a) / s,  torch.cos(a) / s, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="border", align_corners=False)

def train_augment(x):
    x = random_affine(x, 7, 0.05, 0.05)
    B = x.shape[0]
    c = 1 + (torch.rand(B, 1, 1, 1, device=x.device) * 2 - 1) * 0.1
    b = (torch.rand(B, 1, 1, 1, device=x.device) * 2 - 1) * 0.05
    m = x.mean(dim=(2, 3), keepdim=True)
    return ((x - m) * c + m + b).clamp(0, 1)

@torch.no_grad()
def predict_logits(model, V, batch=128, tta_seed=None, amp=True):
    """V: numpy (N,H,W), uint8 or float in [0,1]. Returns float64 logits (N,).
    If tta_seed is given, applies a mild, seeded random affine perturbation (same for every model)."""
    model.eval(); out = []
    gen = None
    if tta_seed is not None:
        gen = torch.Generator(device=DEV); gen.manual_seed(int(tta_seed))
    for i in range(0, len(V), batch):
        xb = torch.from_numpy(np.ascontiguousarray(V[i:i + batch])).to(DEV)
        xb = (xb.float() / 255.0 if xb.dtype == torch.uint8 else xb.float()).unsqueeze(1)
        if gen is not None:
            xb = random_affine(xb, 5, 0.03, 0.03, gen=gen)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            lo = model(to_input(xb)).squeeze(1)
        out.append(lo.float().cpu())
    return torch.cat(out).numpy().astype(np.float64)
print("helpers ready")
from scipy.special import expit
from sklearn.metrics import roc_auc_score
def fit_temperature(logits, y):
    Ts = np.exp(np.linspace(np.log(0.05), np.log(20), 600))
    nll = [-np.mean(y * np.log(expit(logits / T) + 1e-7) + (1 - y) * np.log(1 - expit(logits / T) + 1e-7)) for T in Ts]
    return float(Ts[int(np.argmin(nll))])
def load_original(key):
    m = build(key.rsplit("_s", 1)[0])
    m.load_state_dict(torch.load(os.path.join(ROOT, "ckpt", key + "_best.pt"), map_location=DEV)); m.eval(); return m
def needs_fp32(cname, lvl):          # the two lowest doses overflow fp16 (see paper, Section 2.8)
    return cname == "dose" and lvl >= 4
DEG_CACHE = {}
def degraded(ci):
    if ci not in DEG_CACHE:
        cname, lvl, prm = CONDS[ci]
        DEG_CACHE[ci] = core.apply_condition(V_test, cname, prm, seed=1000 + ci)   # identical to the original evaluation
    return DEG_CACHE[ci]
print("helpers ready")

In [ ]:
# ===== A. R2-a: class-weighted and balanced-sampling training =====
import random
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr).float()
pos_weight = torch.tensor([(ytr == 0).sum() / (ytr == 1).sum()], device=DEV)    # n_neg / n_pos
cls_w = np.where(ytr == 1, 0.5 / ytr.mean(), 0.5 / (1 - ytr.mean()))
for strategy in ["class_weighted", "balanced_sampling"]:
    for name in CFG["MODELS"]:
        for seed in [0, 1, 2]:
            key = f"{strategy}__{name}_s{seed}"
            done = os.path.join(REV, "ckpt", key + "_done.json"); best_p = os.path.join(REV, "ckpt", key + "_best.pt")
            if os.path.exists(done): print("skip", key); continue
            set_seed(seed); model = build(name)
            opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"]); scaler = torch.amp.GradScaler("cuda")
            g = torch.Generator(); g.manual_seed(seed); best, bad = -1, 0
            for ep in range(CFG["EPOCHS"]):
                model.train()
                if strategy == "balanced_sampling":
                    perm = torch.multinomial(torch.from_numpy(cls_w).float(), len(ytr), replacement=True, generator=g)
                else:
                    perm = torch.randperm(len(ytr), generator=g)
                for i in range(0, len(perm), CFG["BATCH"]):
                    idx = perm[i:i + CFG["BATCH"]]
                    xb = train_augment(Xtr_t[idx].to(DEV).float().div_(255).unsqueeze(1)); yb = ytr_t[idx].to(DEV)
                    with torch.autocast("cuda", dtype=torch.float16):
                        out = model(to_input(xb)).squeeze(1).float()
                    loss = F.binary_cross_entropy_with_logits(out, yb, pos_weight=pos_weight if strategy == "class_weighted" else None)
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                sched.step()
                vauc = roc_auc_score(yva, predict_logits(model, Xva))
                if vauc > best: best, bad = vauc, 0; torch.save(model.state_dict(), best_p)
                else: bad += 1
                if bad >= CFG["PATIENCE"]: break
            json.dump(dict(key=key, best_val_auroc=best, epochs=ep + 1), open(done, "w"))
            print(key, "val AUROC", round(best, 4)); del model, opt; torch.cuda.empty_cache()

fp = os.path.join(REV, "preds", "imbalance_logits.npz")
if not os.path.exists(fp):
    out = {}
    for strategy in ["class_weighted", "balanced_sampling"]:
        for name in CFG["MODELS"]:
            for seed in [0, 1, 2]:
                key = f"{strategy}__{name}_s{seed}"
                m = build(name); m.load_state_dict(torch.load(os.path.join(REV, "ckpt", key + "_best.pt"), map_location=DEV)); m.eval()
                out[key + "__T"] = np.array(fit_temperature(predict_logits(m, Xva), yva))
                out[key] = np.stack([predict_logits(m, degraded(ci), amp=not needs_fp32(c, l)) for ci, (c, l, _) in enumerate(CONDS)], 1)
                print("evaluated", key); del m; torch.cuda.empty_cache()
    np.savez_compressed(fp, **out)
Z = dict(np.load(fp)); rows = []
for k, v in Z.items():
    if k.endswith("__T"): continue
    strategy, rest = k.split("__"); name = rest.rsplit("_s", 1)[0]
    for j, (c, l, _) in enumerate(CONDS):
        if l not in (0, 5): continue
        pr = v[:, j] >= 0; err = pr != yte
        rows.append(dict(strategy=strategy, model=name, cond=c, level=l, auroc=roc_auc_score(yte, v[:, j]),
                         sens=pr[yte == 1].mean(), spec=(~pr[yte == 0]).mean(), fp_share=(err & pr).sum() / max(err.sum(), 1)))
A = pd.DataFrame(rows); A.to_csv(os.path.join(REV, "tables", "A_imbalance.csv"), index=False)
print(A.groupby(["strategy", "cond", "level"])[["auroc", "sens", "spec", "fp_share"]].mean().round(3))

In [ ]:
# ===== B. R1-8: Monte Carlo dropout (models that contain dropout layers) =====
NPASS = 8
fp = os.path.join(REV, "preds", "mc_dropout.npz")
if not os.path.exists(fp):
    out = {}
    for name in ["mobilenet_v3_large", "efficientnet_b0"]:          # ResNet-18 has no dropout layer
        for seed in [0, 1, 2]:
            key = f"{name}_s{seed}"; m = load_original(key)
            for mod in m.modules():
                if isinstance(mod, nn.Dropout): mod.train()          # dropout active, BatchNorm stays in eval mode
            arr = np.zeros((len(CONDS), NPASS, 624), dtype=np.float32); t0 = time.time()
            with torch.no_grad():
                for ci, (c, l, _) in enumerate(CONDS):
                    Vd = degraded(ci)
                    for k in range(NPASS):
                        torch.manual_seed(77_000 + 100 * ci + k); outs = []
                        for i in range(0, 624, 128):
                            xb = torch.from_numpy(np.ascontiguousarray(Vd[i:i + 128])).to(DEV).float().unsqueeze(1)
                            with torch.autocast("cuda", dtype=torch.float16, enabled=not needs_fp32(c, l)):
                                outs.append(m(to_input(xb)).squeeze(1).float().cpu())
                        arr[ci, k] = torch.cat(outs).numpy()
            out[key] = arr; out[key + "__seconds"] = np.array(time.time() - t0)
            print("MC dropout", key, f"{time.time() - t0:.0f}s"); del m; torch.cuda.empty_cache()
    np.savez_compressed(fp, **out)
print("MC dropout saved")

In [ ]:
# ===== C. R2-d: test-time augmentation count ablation (seed 0 of each model, K up to 16) =====
KMAX = 16
fp = os.path.join(REV, "preds", "tta_k16.npz")
if not os.path.exists(fp):
    out = {}
    for name in CFG["MODELS"]:
        key = f"{name}_s0"; m = load_original(key)
        arr = np.zeros((len(CONDS), KMAX + 1, 624), dtype=np.float32)
        for ci, (c, l, _) in enumerate(CONDS):
            Vd = degraded(ci); amp = not needs_fp32(c, l)
            arr[ci, 0] = predict_logits(m, Vd, amp=amp)
            for k in range(KMAX):                                    # seeds identical to the original evaluation for k < 8
                arr[ci, k + 1] = predict_logits(m, Vd, tta_seed=10_000 * ci + k, amp=amp)
        out[key] = arr; print("TTA K=16 done", key); del m; torch.cuda.empty_cache()
    np.savez_compressed(fp, **out)
print("TTA ablation saved")

In [ ]:
# ===== D. R1-9: computational cost =====
from torch.utils.flop_counter import FlopCounterMode
fp = os.path.join(REV, "tables", "D_compute.csv")
def gpu_power_sampler(stop, samples):
    while not stop.is_set():
        try:
            w = subprocess.run(["nvidia-smi", "--query-gpu=power.draw", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
            samples.append(float(w.strip().splitlines()[0]))
        except Exception: pass
        time.sleep(0.05)
if not os.path.exists(fp):
    rows = []
    for name in CFG["MODELS"]:
        m = load_original(f"{name}_s0"); n_par = sum(p.numel() for p in m.parameters())
        size_mb = sum(p.numel() * p.element_size() for p in m.parameters()) / 2**20
        mc = build(name).cpu().eval()
        with FlopCounterMode(display=False) as fc:
            with torch.no_grad(): mc(torch.zeros(1, 3, 224, 224))
        gmacs = fc.get_total_flops() / 2 / 1e9
        res = dict(model=name, params_M=n_par / 1e6, size_MB_fp32=size_mb, GMACs=gmacs)
        for prec in ["fp32", "fp16"]:
            x1 = torch.rand(1, 3, 224, 224, device=DEV); x64 = torch.rand(64, 3, 224, 224, device=DEV)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(prec == "fp16")):
                for _ in range(10): m(x1)
                torch.cuda.synchronize(); ts = []
                for _ in range(100):
                    t0 = time.perf_counter(); m(x1); torch.cuda.synchronize(); ts.append(time.perf_counter() - t0)
                torch.cuda.reset_peak_memory_stats(); m(x64); torch.cuda.synchronize()
                peak = torch.cuda.max_memory_allocated() / 2**20
                stop = threading.Event(); samples = []; th = threading.Thread(target=gpu_power_sampler, args=(stop, samples)); th.start()
                t0 = time.perf_counter(); nimg = 0
                while time.perf_counter() - t0 < 10:
                    m(x64); nimg += 64
                torch.cuda.synchronize(); dt = time.perf_counter() - t0; stop.set(); th.join()
            res[f"gpu_{prec}_latency_bs1_ms"] = 1000 * np.median(ts)
            res[f"gpu_{prec}_throughput_img_s"] = nimg / dt
            res[f"gpu_{prec}_peak_mem_bs64_MB"] = peak
            res[f"gpu_{prec}_energy_J_per_img"] = (np.mean(samples) if samples else np.nan) * dt / nimg
            res[f"gpu_{prec}_mean_power_W"] = np.mean(samples) if samples else np.nan
        torch.set_num_threads(2); xc = torch.rand(1, 3, 224, 224); ts = []
        with torch.no_grad():
            for _ in range(5): mc(xc)
            for _ in range(30):
                t0 = time.perf_counter(); mc(xc); ts.append(time.perf_counter() - t0)
        res["cpu_2threads_latency_bs1_ms"] = 1000 * np.median(ts)
        for nm, k in [("tta9", 9), ("ensemble3", 3), ("mcdropout8", 8)]:
            res[f"cpu_2threads_{nm}_latency_ms"] = k * res["cpu_2threads_latency_bs1_ms"]
            res[f"gpu_fp16_{nm}_latency_ms"] = k * res["gpu_fp16_latency_bs1_ms"]
        rows.append(res); print(name, "benchmarked"); del m, mc; torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(fp, index=False)
pd.set_option("display.width", 250); print(pd.read_csv(fp).round(3).T)

In [ ]:
# ===== E. R2-f: EfficientNet-B0 numerical diagnostics at low dose =====
fp = os.path.join(REV, "tables", "E_numerics.csv")
if not os.path.exists(fp):
    rows = []; sub = np.random.default_rng(7).permutation(624)[:64]; Vs = V_test[sub]
    lap = lambda v: np.abs(v[:, 1:-1, 1:-1] * 4 - v[:, :-2, 1:-1] - v[:, 2:, 1:-1] - v[:, 1:-1, :-2] - v[:, 1:-1, 2:]).mean()
    for name in CFG["MODELS"]:
        for seed in [0, 1, 2]:
            key = f"{name}_s{seed}"; m = load_original(key)
            blocks = list(m.features) if hasattr(m, "features") else [m.layer1, m.layer2, m.layer3, m.layer4]
            for d in [1.0, 0.25, 0.1, 0.05, 0.02]:
                Vd = core.deg_dose(Vs, d, np.random.default_rng(50_000 + int(d * 1000)))
                xb = to_input(torch.from_numpy(Vd).to(DEV).float().unsqueeze(1))
                acts = []
                hooks = [b.register_forward_hook(lambda mod, i, o, acts=acts: acts.append((o.float().abs().max().item(), (~torch.isfinite(o)).float().mean().item()))) for b in blocks]
                with torch.no_grad():
                    z32 = m(xb).squeeze(1).float().cpu().numpy()
                for h in hooks: h.remove()
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                    z16 = m(xb).squeeze(1).float().cpu().numpy()
                md = m.double()
                with torch.no_grad():
                    z64 = md(xb.double()).squeeze(1).cpu().numpy()
                m.float()
                rows.append(dict(model=name, seed=seed, dose=d, input_min=Vd.min(), input_max=Vd.max(), input_mean=Vd.mean(), input_laplacian=lap(Vd),
                                 max_abs_activation=max(a for a, _ in acts), block_of_max=int(np.argmax([a for a, _ in acts])),
                                 nonfinite_frac_fp16=float(np.mean(~np.isfinite(z16))), max_abs_logit_fp32=float(np.abs(z32).max()),
                                 max_abs_logit_fp64=float(np.abs(z64).max()), max_rel_diff_fp32_fp64=float(np.max(np.abs(z32 - z64) / (np.abs(z64) + 1e-6)))))
            print("diagnosed", key); del m; torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(fp, index=False)
E = pd.read_csv(fp); print(E.groupby(["model", "dose"]).mean(numeric_only=True).drop(columns="seed").round(4))

In [ ]:
# ===== Package results for upload to the chat =====
import shutil
pkg = "/content/revision_results"
if os.path.exists(pkg): shutil.rmtree(pkg)
shutil.copytree(REV, pkg, ignore=shutil.ignore_patterns("*.pt"))
zp = shutil.make_archive(os.path.join(ROOT, "revision_results"), "zip", pkg)
print("Upload this file to the chat:", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")